In [1]:
# =========================
# CELL 1: Smart offline install (minimal + compatible wheels only)
# =========================
import os, glob, subprocess, sys, re

def _run(cmd):
    print(">>", " ".join(cmd))
    return subprocess.run(cmd, check=False, capture_output=True, text=True)

# Kaggle usually: Python 3.11
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
print("🐍 Python tag:", py_tag)

# Find wheels inside any /kaggle/input/** directory
wheel_paths = glob.glob("/kaggle/input/**/**/*.whl", recursive=True)
print("📦 Found wheels:", len(wheel_paths))

# Keep only wheels compatible with current python tag
compatible = []
for w in wheel_paths:
    name = os.path.basename(w)
    if py_tag in name and ("manylinux" in name or "linux" in name):
        compatible.append(w)

# Prefer installing only what we need (segmentation_models_pytorch, ultralytics if missing)
need = ["segmentation_models_pytorch", "ultralytics"]
to_install = []

import importlib.util
for pkg in need:
    if importlib.util.find_spec(pkg) is None:
        # choose best matching wheel
        candidates = [w for w in compatible if re.search(rf"(^|[-_]){pkg}([-.]|_)", os.path.basename(w), re.I)]
        if candidates:
            to_install.append(candidates[0])

print("✅ Will try install:", len(to_install), "wheels")
ok = 0
for w in to_install:
    r = _run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", w])
    if r.returncode == 0:
        ok += 1
    else:
        print("⚠️ Failed:", os.path.basename(w))
        print(r.stderr[:300])

print(f"✅ Installed {ok}/{len(to_install)} wheels safely.")


🐍 Python tag: cp311
📦 Found wheels: 190
✅ Will try install: 0 wheels
✅ Installed 0/0 wheels safely.


In [2]:
# =========================
# CELL 2: Imports + Config + Auto-Find Inputs
# =========================
import os, glob, re, gc
import numpy as np
import pandas as pd

import torch

# These should exist after Cell 1, but Kaggle often has them preinstalled
import segmentation_models_pytorch as smp
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("⚡ Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# --- Controls ---
USE_UNET = True                 # True: UNet mask, False: pure CV extraction
USE_OLD_FALLBACK = True         # if NEW UNet mask looks bad, try OLD
YOLO_CONF = 0.25
YOLO_IOU  = 0.45

# --- Expected datasets (your ones) ---
NEW_UNET_PATH = "/kaggle/input/best-dd4/best_model_effb3_phase9_ddp.pth"
OLD_UNET_PATH = "/kaggle/input/ecg-final-models1/best_model_phase10.pth"
YOLO_PATH     = "/kaggle/input/ecg-final-models1/best.pt"

print("📌 Paths:")
print(" - NEW_UNET_PATH:", NEW_UNET_PATH, "| exists:", os.path.exists(NEW_UNET_PATH))
print(" - OLD_UNET_PATH:", OLD_UNET_PATH, "| exists:", os.path.exists(OLD_UNET_PATH))
print(" - YOLO_PATH    :", YOLO_PATH,     "| exists:", os.path.exists(YOLO_PATH))

assert os.path.exists(NEW_UNET_PATH), "❌ NEW_UNET_PATH not found"
assert os.path.exists(OLD_UNET_PATH), "❌ OLD_UNET_PATH not found"
assert os.path.exists(YOLO_PATH),     "❌ YOLO_PATH not found"


def find_first(patterns):
    """Return first matching file path for any pattern under /kaggle/input."""
    for pat in patterns:
        hits = glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
        if hits:
            return hits[0]
    return None

TEST_CSV = find_first(["test.csv", "*test*.csv"])
SAMPLE_SUB = find_first(["sample_submission.csv", "*sample*submission*.csv"])

print("📄 Found test.csv:", TEST_CSV)
print("📄 Found sample_submission.csv:", SAMPLE_SUB)

assert SAMPLE_SUB is not None, "❌ Could not find sample_submission.csv in /kaggle/input"

sample_sub = pd.read_csv(SAMPLE_SUB)
print("✅ sample_submission shape:", sample_sub.shape)
print("✅ sample_submission columns:", list(sample_sub.columns)[:20])

# Try to locate test.csv; if not found, we will infer inputs from sample_submission
test_df = None
if TEST_CSV is not None:
    test_df = pd.read_csv(TEST_CSV)
    print("✅ test.csv shape:", test_df.shape)
    print("✅ test.csv columns:", list(test_df.columns))
else:
    print("⚠️ test.csv not found; will use sample_submission as base (best effort).")


ModuleNotFoundError: No module named 'segmentation_models_pytorch'

In [ ]:
# =========================
# CELL 3: Index images (auto)
# =========================
import os, glob

IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp")

# Find likely image folders (large collections of images)
all_imgs = []
for ext in IMG_EXTS:
    all_imgs.extend(glob.glob(f"/kaggle/input/**/*{ext}", recursive=True))

print("🗂️ Total images found in /kaggle/input:", len(all_imgs))
assert len(all_imgs) > 0, "❌ No images found under /kaggle/input"

# Build basename->path map
basename_to_path = {}
for p in all_imgs:
    base = os.path.splitext(os.path.basename(p))[0]
    # Keep first occurrence
    basename_to_path.setdefault(base, p)

print("✅ Unique basenames indexed:", len(basename_to_path))

# Try to check coverage with ids from test_df or sample_sub
def get_id_series():
    if test_df is not None and "id" in test_df.columns:
        return test_df["id"].astype(str)
    if "id" in sample_sub.columns:
        return sample_sub["id"].astype(str)
    # fallback: first column
    return sample_sub.iloc[:,0].astype(str)

ids = get_id_series()
ids_sample = ids.sample(min(200, len(ids)), random_state=42).tolist()

resolved = 0
for x in ids_sample:
    if x in basename_to_path:
        resolved += 1
    else:
        # sometimes id has suffix/prefix: try base extraction
        x2 = str(x).split(".")[0]
        if x2 in basename_to_path:
            resolved += 1

print(f"🧪 Sanity resolve: {resolved}/{len(ids_sample)} = {100*resolved/len(ids_sample):.1f}%")
print("✅ Cell 3 ready.")


In [ ]:
# =========================
# CELL 4: Load YOLO + Smart UNet auto loader
# =========================
def _unwrap_state_dict(obj):
    if isinstance(obj, dict):
        if "state_dict" in obj and isinstance(obj["state_dict"], dict):
            obj = obj["state_dict"]
        if "model" in obj and isinstance(obj["model"], dict):
            obj = obj["model"]
    if isinstance(obj, dict) and any(k.startswith("module.") for k in obj.keys()):
        obj = {k.replace("module.", "", 1): v for k, v in obj.items()}
    return obj

def _guess_encoder(sd, path_hint=""):
    keys = list(sd.keys())

    # EfficientNet timm pattern
    if any(k.startswith("encoder._conv_stem") or k.startswith("encoder._blocks") for k in keys):
        # use hint from filename
        if "effb3" in path_hint.lower() or "b3" in path_hint.lower():
            return "timm-efficientnet-b3"
        return "timm-efficientnet-b3"

    # ResNet pattern
    if any(k.startswith("encoder.conv1") or k.startswith("encoder.layer1") for k in keys):
        return "resnet50"

    return "resnet50"

def build_unet(encoder_name: str):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights=None,
        in_channels=3,
        classes=1,
        decoder_attention_type="scse",
    )

def load_unet_auto(path: str):
    ckpt = torch.load(path, map_location="cpu")
    sd = _unwrap_state_dict(ckpt)
    enc = _guess_encoder(sd, path_hint=os.path.basename(path))

    model = build_unet(enc).to(DEVICE).eval()
    missing, unexpected = model.load_state_dict(sd, strict=False)

    score = len(missing) + len(unexpected)
    print(f"✅ Loaded: {os.path.basename(path)} | encoder={enc} | missing={len(missing)} unexpected={len(unexpected)}")

    # If mismatch big, brute try common encoders
    if score > 200:
        best = (score, enc, model)
        for alt in ["timm-efficientnet-b0","timm-efficientnet-b1","timm-efficientnet-b2","timm-efficientnet-b3","resnet50"]:
            try:
                m2 = build_unet(alt).to(DEVICE).eval()
                mi2, un2 = m2.load_state_dict(sd, strict=False)
                s2 = len(mi2) + len(un2)
                if s2 < best[0]:
                    best = (s2, alt, m2)
            except:
                pass
        score, enc, model = best
        print(f"   🔁 Final encoder={enc} | mismatch_score={score}")

    return model

print("⚙️ Loading YOLO...")
yolo = YOLO(YOLO_PATH)
print("✅ YOLO loaded:", YOLO_PATH)

print("⚙️ Loading UNets (auto-detect)...")
new_model = load_unet_auto(NEW_UNET_PATH)
old_model = load_unet_auto(OLD_UNET_PATH)

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

print("✅ Models ready.")


In [ ]:
# =========================
# CELL 5: Utils (crop leads + extract trace)
# =========================
import cv2
from contextlib import contextmanager

# YOLO class -> lead index (0..11)
CLASS_TO_LEAD_IDX = {i:i for i in range(12)}

@contextmanager
def autocast_ctx():
    if DEVICE == "cuda":
        with torch.amp.autocast(device_type="cuda", enabled=True):
            yield
    else:
        yield

def read_image(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def fallback_split_12leads(img):
    """Fallback crop into 12 boxes (4x3) from whole ECG sheet."""
    H, W = img.shape[:2]
    rows, cols = 4, 3
    crops = {}
    h = H // rows
    w = W // cols
    idx = 0
    for r in range(rows):
        for c in range(cols):
            y1 = r*h
            y2 = (r+1)*h if r < rows-1 else H
            x1 = c*w
            x2 = (c+1)*w if c < cols-1 else W
            crops[idx] = img[y1:y2, x1:x2].copy()
            idx += 1
    return crops  # 0..11

def yolo_lead_crops(img):
    """Run YOLO once, return dict lead_idx -> crop."""
    H, W = img.shape[:2]
    crops = {}
    try:
        res = yolo.predict(img, conf=YOLO_CONF, iou=YOLO_IOU, verbose=False)
        boxes = res[0].boxes
        if boxes is None or len(boxes) == 0:
            return None

        # Each box has cls and xyxy
        for b in boxes:
            cls = int(b.cls.item())
            if cls not in CLASS_TO_LEAD_IDX:
                continue
            lead_idx = CLASS_TO_LEAD_IDX[cls]
            x1,y1,x2,y2 = b.xyxy[0].tolist()
            x1 = max(0, int(x1)); y1 = max(0, int(y1))
            x2 = min(W, int(x2)); y2 = min(H, int(y2))
            if x2-x1 < 10 or y2-y1 < 10:
                continue
            crops[lead_idx] = img[y1:y2, x1:x2].copy()

        if len(crops) == 0:
            return None
        return crops
    except Exception as e:
        return None

def estimate_small_box_px(gray):
    """
    Estimate small grid spacing in pixels using projection autocorrelation.
    Returns small_box_px or None.
    """
    try:
        g = gray.copy()
        g = cv2.GaussianBlur(g, (3,3), 0)
        # emphasize grid lines (edges)
        sx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
        sy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
        px = np.mean(np.abs(sx), axis=0)  # per x
        py = np.mean(np.abs(sy), axis=1)  # per y

        def peak_period(p):
            p = p - p.mean()
            ac = np.correlate(p, p, mode="full")
            ac = ac[len(ac)//2:]
            ac[:5] = 0
            # find first strong peak
            k = np.argmax(ac[:200])  # search up to 200 px
            if ac[k] <= 0:
                return None
            return int(k) if k > 5 else None

        per_x = peak_period(px)
        per_y = peak_period(py)

        cand = [v for v in [per_x, per_y] if v is not None]
        if not cand:
            return None
        # grid is usually same spacing both directions
        return int(np.median(cand))
    except:
        return None

def unet_mask(model, crop_rgb, out_hw=(256, 512)):
    """Return binary mask for trace."""
    h, w = out_hw
    img = cv2.resize(crop_rgb, (w, h), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    x = torch.from_numpy(img).permute(2,0,1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with autocast_ctx():
            pred = model(x)
            pred = torch.sigmoid(pred).float().squeeze().detach().cpu().numpy()

    mask = (pred > 0.5).astype(np.uint8)
    return mask  # (h,w)

def cv_mask(crop_rgb, out_hw=(256, 512)):
    """Fast CV mask fallback."""
    h, w = out_hw
    g = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY)
    g = cv2.resize(g, (w,h), interpolation=cv2.INTER_AREA)

    # remove grid using morphological opening
    blur = cv2.GaussianBlur(g, (3,3), 0)
    # adaptive threshold (trace tends to be darker)
    th = cv2.adaptiveThreshold(blur, 255, cv2.ADAPTIVE_THRESH_MEAN_C,
                              cv2.THRESH_BINARY_INV, 31, 7)

    # cleanup
    th = cv2.medianBlur(th, 3)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, np.ones((2,2), np.uint8), iterations=1)
    return (th > 0).astype(np.uint8)

def mask_to_wave(mask):
    """Convert mask (h,w) -> y(x) curve, returns float array length w."""
    h, w = mask.shape
    ys = np.full(w, np.nan, dtype=np.float32)

    # For each x, find median y of active pixels
    for x in range(w):
        col = np.where(mask[:, x] > 0)[0]
        if len(col) > 0:
            ys[x] = np.median(col)

    # fill gaps
    if np.all(np.isnan(ys)):
        return None

    # interpolate nans
    x = np.arange(w)
    good = ~np.isnan(ys)
    ys = np.interp(x, x[good], ys[good]).astype(np.float32)

    # smooth
    # simple moving average to keep dependencies minimal
    k = 9
    ys = np.convolve(ys, np.ones(k)/k, mode="same").astype(np.float32)
    return ys

def build_signal_from_crop(crop_rgb, target_len, fs=None):
    """
    Extract waveform from a lead crop.
    Returns float32 array length target_len.
    """
    out_h, out_w = 256, 512

    # grid scale estimation for amplitude
    gray = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2GRAY)
    gray_small = cv2.resize(gray, (out_w, out_h), interpolation=cv2.INTER_AREA)
    small_box = estimate_small_box_px(gray_small)
    # default guess if cannot estimate
    if small_box is None or small_box < 3 or small_box > 80:
        small_box = 10  # safe fallback

    px_per_mV = small_box * 10.0  # 10mm per mV (10 small boxes)
    # mask extraction
    if USE_UNET:
        m = unet_mask(new_model, crop_rgb, out_hw=(out_h,out_w))
        ys = mask_to_wave(m)
        # If too empty, fallback to OLD or CV
        if ys is None or np.mean(m) < 0.001:
            if USE_OLD_FALLBACK:
                m2 = unet_mask(old_model, crop_rgb, out_hw=(out_h,out_w))
                ys2 = mask_to_wave(m2)
                if ys2 is not None and np.mean(m2) >= np.mean(m):
                    ys = ys2
                    m = m2
            if ys is None or np.mean(m) < 0.001:
                m = cv_mask(crop_rgb, out_hw=(out_h,out_w))
                ys = mask_to_wave(m)
    else:
        m = cv_mask(crop_rgb, out_hw=(out_h,out_w))
        ys = mask_to_wave(m)

    if ys is None:
        return np.zeros(int(target_len), dtype=np.float32)

    # convert y(x) -> amplitude in mV (centered)
    mid = np.median(ys)
    amp_mV = (mid - ys) / float(px_per_mV)  # invert y
    # robust scale clamp
    amp_mV = np.clip(amp_mV, -3.0, 3.0).astype(np.float32)

    # resample to target_len
    x_src = np.linspace(0, 1, len(amp_mV), dtype=np.float32)
    x_tgt = np.linspace(0, 1, int(target_len), dtype=np.float32)
    sig = np.interp(x_tgt, x_src, amp_mV).astype(np.float32)

    # final clamp
    sig = np.clip(sig, -3.5, 3.5).astype(np.float32)
    return sig

def resolve_image_path(sample_id: str):
    """Resolve id -> image path using basename_to_path; best effort."""
    s = str(sample_id)
    if s in basename_to_path:
        return basename_to_path[s]
    s2 = s.split(".")[0]
    if s2 in basename_to_path:
        return basename_to_path[s2]
    return None


In [ ]:
# =========================
# CELL 6: Build submission based on sample_submission format
# =========================
# Determine base dataframe for rows
base = None
if test_df is not None and all(c in test_df.columns for c in ["id","lead"]):
    base = test_df.copy()
else:
    # fallback: assume sample has the required rows/ordering
    base = sample_sub.copy()

assert "id" in base.columns, "❌ No 'id' column found in base data."
assert "lead" in base.columns, "❌ No 'lead' column found in base data."

# Identify target columns in sample_submission (columns not in test columns)
# If sample has a known target column, we'll fill it.
key_cols = [c for c in ["id","lead","fs","number_of_rows"] if c in sample_sub.columns]
target_cols = [c for c in sample_sub.columns if c not in key_cols]
if len(target_cols) == 0:
    # sometimes sample has same columns as test and expects additional hidden format;
    # we'll create a 'signal' column.
    target_cols = ["signal"]
    sample_sub = base[["id","lead"]].copy()
    sample_sub["signal"] = ""
else:
    sample_sub = sample_sub.copy()

print("🔎 Key cols:", key_cols)
print("🎯 Target cols:", target_cols)

# We need fs and number_of_rows. If missing, create defaults (best effort)
if "fs" not in base.columns:
    base["fs"] = 500  # fallback
if "number_of_rows" not in base.columns:
    base["number_of_rows"] = 1000  # fallback

# Helper to format prediction like example in sample_submission
def format_like_example(example, arr_or_val):
    # if numeric target
    if isinstance(example, (int, float, np.integer, np.floating)):
        if isinstance(arr_or_val, (list, np.ndarray)):
            return float(np.mean(arr_or_val))
        return float(arr_or_val)

    # string target: keep small output (you can switch to dense if needed)
    # We'll output space-separated with 4 decimals
    if isinstance(arr_or_val, (list, np.ndarray)):
        a = np.asarray(arr_or_val, dtype=np.float32)
        return " ".join([f"{x:.4f}" for x in a.tolist()])
    return str(arr_or_val)

# --- Main caching: process per unique image id ---
unique_ids = base["id"].astype(str).unique().tolist()
print("🧠 Unique ids:", len(unique_ids))

# cache: (id)-> dict lead_str->signal
cache = {}

def lead_to_idx(lead_val):
    # If lead is numeric 0..11
    try:
        lv = int(lead_val)
        if 0 <= lv <= 11:
            return lv
    except:
        pass
    # If lead is string like "I","II"... we map common:
    m = {
        "I":0, "II":1, "III":2, "aVR":3, "aVL":4, "aVF":5,
        "V1":6, "V2":7, "V3":8, "V4":9, "V5":10, "V6":11
    }
    s = str(lead_val).strip()
    return m.get(s, 0)

# progress
from tqdm import tqdm

for sid in tqdm(unique_ids, desc="Processing images"):
    path = resolve_image_path(sid)
    if path is None:
        cache[sid] = {}
        continue

    img = read_image(path)
    if img is None:
        cache[sid] = {}
        continue

    # YOLO crops; fallback split if fails
    crops = yolo_lead_crops(img)
    if crops is None or len(crops) < 6:
        crops = fallback_split_12leads(img)

    cache[sid] = {"__crops__": crops}  # store crops only once

# Build submission rows
out = sample_sub.copy()

# Choose example values for target formatting
example_map = {}
for c in target_cols:
    ex = out[c].iloc[0] if c in out.columns and len(out) else ""
    example_map[c] = ex

pred_values = {c: [] for c in target_cols}

for i in tqdm(range(len(base)), desc="Building rows"):
    sid = str(base.loc[i, "id"])
    lead_val = base.loc[i, "lead"]
    fs = float(base.loc[i, "fs"])
    n = int(base.loc[i, "number_of_rows"])

    crops = cache.get(sid, {}).get("__crops__", None)
    if crops is None:
        sig = np.zeros(n, dtype=np.float32)
    else:
        lidx = lead_to_idx(lead_val)
        crop = crops.get(lidx, None)
        if crop is None:
            sig = np.zeros(n, dtype=np.float32)
        else:
            sig = build_signal_from_crop(crop, target_len=n, fs=fs)

    # If there is exactly 1 target column, fill it
    if len(target_cols) == 1:
        c = target_cols[0]
        pred_values[c].append(format_like_example(example_map[c], sig))
    else:
        # If multiple targets, fill each with scalar/stat (best effort)
        for c in target_cols:
            pred_values[c].append(format_like_example(example_map[c], float(np.mean(sig))))

# Assign to out (ensure same length)
for c in target_cols:
    if c not in out.columns:
        out[c] = ""
    out[c] = pred_values[c]

print("✅ Built submission frame:", out.shape)
print(out.head(3))


In [ ]:
# =========================
# CELL 7: Validate + Write submission.csv
# =========================
SUB_PATH = "submission.csv"

# Basic checks
assert len(out) == len(sample_sub), "❌ submission row count mismatch"
assert all(c in out.columns for c in sample_sub.columns), "❌ submission columns mismatch"

# No NaNs in targets
for c in target_cols:
    if out[c].isna().any():
        out[c] = out[c].fillna("0")

out.to_csv(SUB_PATH, index=False)
print("✅ Done. submission.csv ready.")
print("📦 submission.csv size:", round(os.path.getsize(SUB_PATH)/1024/1024, 2), "MB")

# quick preview
print("✅ Columns:", list(out.columns))
print(out.head(2))


In [ ]:
# =========================
# CELL 8 (Optional): Quick debug on one id
# =========================
import random
sid = str(base["id"].iloc[random.randint(0, len(base)-1)])
path = resolve_image_path(sid)
print("Sample id:", sid)
print("Image path:", path)

if path is not None:
    img = read_image(path)
    crops = yolo_lead_crops(img) or fallback_split_12leads(img)
    # show one crop stats
    c0 = crops.get(0, None)
    if c0 is not None:
        sig = build_signal_from_crop(c0, target_len=1000, fs=500)
        print("Sig preview:", sig[:10], "min/max:", float(sig.min()), float(sig.max()))
